In [1]:
pip install "numpy<2.0" --force-reinstall --break-system-packages

  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
deepnote-toolkit 1.1.2 requires pandas<2.2,>=1.2.5; python_version < "3.12", but you have pandas 2.3.3 which is incompatible.

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
#!/usr/bin/env python3
"""
4_VISUALIZATIONS_REVISED_MPL39.py

Creates publication-quality visualizations for LLM pluralistic ignorance predictions.
- Fixed the mae_ensemble error
- Removed figure titles
- Shows MAE instead of predictions
- COMPATIBLE WITH MATPLOTLIB 3.9+
"""

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

# Set publication-quality style (Matplotlib 3.9+ compatible)
# Use seaborn's newer API or configure matplotlib directly
import matplotlib.pyplot as plt

# Configure matplotlib directly for better compatibility
matplotlib.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 13
})

# Use seaborn style without set_context (which has compatibility issues)
sns.set_style("whitegrid")

print("="*80)
print("VISUALIZATION SCRIPT - MATPLOTLIB 3.9+ COMPATIBLE")
print("="*80)

# Load data
df = pd.read_csv("predictions_all_stages_long.csv")
model_cols = [col for col in df.columns if col.startswith('pred_')]
df['pred_ensemble'] = df[model_cols].mean(axis=1)

# Add continent mapping
continent_mapping = {
    'Afghanistan': 'Asia', 'Albania': 'Europe', 'Algeria': 'Africa', 'Argentina': 'South America',
    'Armenia': 'Asia', 'Australia': 'Oceania', 'Austria': 'Europe', 'Bangladesh': 'Asia',
    'Belgium': 'Europe', 'Benin': 'Africa', 'Bolivia': 'South America', 'Bosnia Herzegovina': 'Europe',
    'Botswana': 'Africa', 'Brazil': 'South America', 'Bulgaria': 'Europe', 'Burkina Faso': 'Africa',
    'Cambodia': 'Asia', 'Cameroon': 'Africa', 'Canada': 'North America', 'Chad': 'Africa',
    'Chile': 'South America', 'China': 'Asia', 'Colombia': 'South America', 'Congo Brazzaville': 'Africa',
    'Costa Rica': 'North America', 'Croatia': 'Europe', 'Cyprus': 'Europe', 'Czech Republic': 'Europe',
    'Denmark': 'Europe', 'Dominican Republic': 'North America', 'Ecuador': 'South America', 'Egypt': 'Africa',
    'El Salvador': 'North America', 'Estonia': 'Europe', 'Ethiopia': 'Africa', 'Finland': 'Europe',
    'France': 'Europe', 'Gabon': 'Africa', 'Georgia': 'Asia', 'Germany': 'Europe',
    'Ghana': 'Africa', 'Greece': 'Europe', 'Guatemala': 'North America', 'Guinea': 'Africa',
    'Haiti': 'North America', 'Honduras': 'North America', 'Hong Kong': 'Asia', 'Hungary': 'Europe',
    'Iceland': 'Europe', 'India': 'Asia', 'Indonesia': 'Asia', 'Iran': 'Asia',
    'Iraq': 'Asia', 'Ireland': 'Europe', 'Israel': 'Asia', 'Italy': 'Europe',
    'Ivory Coast': 'Africa', 'Jamaica': 'North America', 'Japan': 'Asia', 'Jordan': 'Asia',
    'Kazakhstan': 'Asia', 'Kenya': 'Africa', 'Kosovo': 'Europe', 'Kyrgyzstan': 'Asia',
    'Laos': 'Asia', 'Latvia': 'Europe', 'Lebanon': 'Asia', 'Liberia': 'Africa', 'Libya': 'Africa',
    'Lithuania': 'Europe', 'Luxembourg': 'Europe', 'Macedonia': 'Europe', 'Madagascar': 'Africa',
    'Malawi': 'Africa', 'Malaysia': 'Asia', 'Mali': 'Africa', 'Malta': 'Europe',
    'Mauritania': 'Africa', 'Mauritius': 'Africa', 'Mexico': 'North America', 'Moldova': 'Europe',
    'Mongolia': 'Asia', 'Montenegro': 'Europe', 'Morocco': 'Africa', 'Mozambique': 'Africa',
    'Myanmar': 'Asia', 'Namibia': 'Africa', 'Nepal': 'Asia', 'Netherlands': 'Europe',
    'New Zealand': 'Oceania', 'Nicaragua': 'North America', 'Niger': 'Africa', 'Nigeria': 'Africa',
    'North Macedonia': 'Europe', 'Norway': 'Europe', 'Pakistan': 'Asia', 'Palestinian Territories': 'Asia',
    'Panama': 'North America', 'Paraguay': 'South America', 'Peru': 'South America', 'Philippines': 'Asia',
    'Poland': 'Europe', 'Portugal': 'Europe', 'Romania': 'Europe', 'Russia': 'Europe', 'Rwanda': 'Africa',
    'Saudi Arabia': 'Asia', 'Senegal': 'Africa', 'Serbia': 'Europe', 'Sierra Leone': 'Africa',
    'Singapore': 'Asia', 'Slovakia': 'Europe', 'Slovenia': 'Europe', 'South Africa': 'Africa',
    'South Korea': 'Asia', 'Spain': 'Europe', 'Sri Lanka': 'Asia', 'Sweden': 'Europe',
    'Switzerland': 'Europe', 'Taiwan': 'Asia', 'Tajikistan': 'Asia', 'Tanzania': 'Africa',
    'Thailand': 'Asia', 'Togo': 'Africa', 'Tunisia': 'Africa', 'Turkey': 'Asia',
    'Turkmenistan': 'Asia', 'Uganda': 'Africa', 'Ukraine': 'Europe', 'United Arab Emirates': 'Asia',
    'United Kingdom': 'Europe', 'United States': 'North America', 'Uruguay': 'South America',
    'Uzbekistan': 'Asia', 'Venezuela': 'South America', 'Vietnam': 'Asia', 'Yemen': 'Asia',
    'Zambia': 'Africa', 'Zimbabwe': 'Africa'
}
df['continent'] = df['countrynew'].map(continent_mapping)

# Try to load ground truth
try:
    gt_df = pd.read_csv("data_final.csv")
    if 'mean_other_willingness' in gt_df.columns:
        gt_df['ground_truth_pi'] = gt_df['mean_other_willingness'] * 100
        df = df.merge(gt_df[['countrynew', 'ground_truth_pi', 'mean_own_willingness']], 
                     on='countrynew', how='left')
        has_ground_truth = True
    else:
        has_ground_truth = False
except:
    has_ground_truth = False

print(f"Ground truth available: {has_ground_truth}")

# Calculate MAE for each model and ensemble
if has_ground_truth:
    for col in model_cols:
        df[f'mae_{col.replace("pred_", "")}'] = abs(df[col] - df['ground_truth_pi'])
    df['mae_ensemble'] = abs(df['pred_ensemble'] - df['ground_truth_pi'])
    mae_cols = [col for col in df.columns if col.startswith('mae_')]
    print(f"Created MAE columns: {mae_cols}")

# ================================================================
# Figure 1: Stage Effects (now showing MAE)
# ================================================================

print("\nCreating Figure 1: Stage Effects...")

# Matplotlib 3.9+ compatible: Use fig.set_size_inches() and fig.set_dpi()
fig = plt.figure()
fig.set_size_inches(14, 10)
fig.set_dpi(300)
axes = fig.subplots(2, 2)

if has_ground_truth:
    # 1a: Mean MAE by stage for each model
    ax = axes[0, 0]
    mae_model_cols = [col for col in df.columns if col.startswith('mae_') and col != 'mae_ensemble']
    stage_mae = df.groupby('stage')[mae_model_cols].mean()
    stage_mae.columns = [col.replace('mae_', '').upper() for col in stage_mae.columns]
    
    for col in stage_mae.columns:
        ax.plot(stage_mae.index, stage_mae[col], marker='o', label=col, linewidth=2)
    
    ax.axhline(y=df['mae_ensemble'].mean(), color='gray', linestyle='--', alpha=0.5, label='Ensemble Mean')
    ax.set_xlabel('Stage', fontweight='bold')
    ax.set_ylabel('Mean Absolute Error (pp)', fontweight='bold')
    ax.set_title('(a) Mean Absolute Error by Stage', fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    # 1b: MAE variance by stage
    ax = axes[0, 1]
    stage_mae_std = df.groupby('stage')[mae_model_cols].std()
    stage_mae_std.columns = [col.replace('mae_', '').upper() for col in stage_mae_std.columns]
    
    for col in stage_mae_std.columns:
        ax.plot(stage_mae_std.index, stage_mae_std[col], marker='s', label=col, linewidth=2)
    
    ax.set_xlabel('Stage', fontweight='bold')
    ax.set_ylabel('Standard Deviation of MAE (pp)', fontweight='bold')
    ax.set_title('(b) MAE Variability by Stage', fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    # 1c: Box plot of ensemble MAE by stage
    ax = axes[1, 0]
    stage_data = [df[df['stage'] == s]['mae_ensemble'].values for s in range(1, 9)]
    bp = ax.boxplot(stage_data, labels=range(1, 9), patch_artist=True)
    
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    
    ax.set_xlabel('Stage', fontweight='bold')
    ax.set_ylabel('Mean Absolute Error (pp)', fontweight='bold')
    ax.set_title('(c) Distribution of Ensemble MAE by Stage', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
else:
    # If no ground truth, show predictions instead
    ax = axes[0, 0]
    stage_means = df.groupby('stage')[model_cols].mean()
    stage_means.columns = [col.replace('pred_', '').upper() for col in stage_means.columns]
    
    for col in stage_means.columns:
        ax.plot(stage_means.index, stage_means[col], marker='o', label=col, linewidth=2)
    
    ax.set_xlabel('Stage', fontweight='bold')
    ax.set_ylabel('Mean Prediction (%)', fontweight='bold')
    ax.set_title('(a) Mean Predictions by Stage (No Ground Truth)', fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    ax = axes[0, 1]
    ax.text(0.5, 0.5, 'Ground truth not available\nfor MAE calculation', 
            ha='center', va='center', fontsize=12, transform=ax.transAxes)
    ax.axis('off')
    
    ax = axes[1, 0]
    ax.text(0.5, 0.5, 'Ground truth not available\nfor MAE calculation', 
            ha='center', va='center', fontsize=12, transform=ax.transAxes)
    ax.axis('off')

# 1d: Stage information labels
ax = axes[1, 1]
ax.axis('off')
stage_info = [
    "Stage 1: Country only",
    "Stage 2: + Socio-demographics",
    "Stage 3: + Macro-economics",
    "Stage 4: + Temperature",
    "Stage 5: + Own willingness",
    "Stage 6: + Socio + Macro",
    "Stage 7: + Socio + Macro + Temp",
    "Stage 8: + All information"
]

y_start = 0.9
for i, info in enumerate(stage_info):
    ax.text(0.1, y_start - (i * 0.11), info, fontsize=11, verticalalignment='top')

ax.set_title('(d) Information by Stage', fontweight='bold')

plt.tight_layout()
plt.savefig('figure1_stage_effects.png', dpi=300, bbox_inches='tight')
plt.savefig('figure1_stage_effects.pdf', dpi=300, bbox_inches='tight')
print("✓ Saved: figure1_stage_effects.png")
print("✓ Saved: figure1_stage_effects.pdf")
plt.close()

# ================================================================
# Figure 2: Model Comparison (showing MAE)
# ================================================================

print("\nCreating Figure 2: Model Comparison...")

fig = plt.figure()
fig.set_size_inches(14, 10)
fig.set_dpi(300)
axes = fig.subplots(2, 2)

if has_ground_truth:
    # 2a: Overall MAE by model
    ax = axes[0, 0]
    overall_mae = df[mae_model_cols + ['mae_ensemble']].mean().sort_values()
    overall_mae.index = [idx.replace('mae_', '').upper() for idx in overall_mae.index]
    
    colors = ['lightcoral' if 'ENSEMBLE' in idx else 'lightblue' for idx in overall_mae.index]
    bars = ax.barh(range(len(overall_mae)), overall_mae.values, color=colors, alpha=0.7)
    ax.set_yticks(range(len(overall_mae)))
    ax.set_yticklabels(overall_mae.index)
    ax.set_xlabel('Mean Absolute Error (pp)', fontweight='bold')
    ax.set_title('(a) Overall MAE by Model', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for i, (idx, val) in enumerate(overall_mae.items()):
        ax.text(val, i, f' {val:.2f}', va='center', fontweight='bold')
    
    # 2b: MAE by stage for top models
    ax = axes[0, 1]
    top_models = overall_mae.head(4).index
    stage_mae_top = df.groupby('stage')[[f"mae_{m.lower()}" for m in top_models if f"mae_{m.lower()}" in df.columns]].mean()
    
    for col in stage_mae_top.columns:
        model_name = col.replace('mae_', '').upper()
        ax.plot(stage_mae_top.index, stage_mae_top[col], marker='o', label=model_name, linewidth=2)
    
    ax.set_xlabel('Stage', fontweight='bold')
    ax.set_ylabel('Mean Absolute Error (pp)', fontweight='bold')
    ax.set_title('(b) Top Models MAE Across Stages', fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    # 2c: Model agreement (correlation between models' MAE)
    ax = axes[1, 0]
    mae_corr = df[mae_model_cols].corr()
    mae_corr.index = [idx.replace('mae_', '').upper() for idx in mae_corr.index]
    mae_corr.columns = [col.replace('mae_', '').upper() for col in mae_corr.columns]
    
    sns.heatmap(mae_corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
                ax=ax, cbar_kws={'label': 'Correlation'}, vmin=-1, vmax=1)
    ax.set_title('(c) Model MAE Correlation Matrix', fontweight='bold')
    
    # 2d: Distribution of MAE across all predictions
    ax = axes[1, 1]
    ax.hist(df['mae_ensemble'], bins=30, alpha=0.7, color='lightblue', edgecolor='black')
    ax.axvline(df['mae_ensemble'].mean(), color='red', linestyle='--', linewidth=2, 
               label=f'Mean: {df["mae_ensemble"].mean():.2f}')
    ax.axvline(df['mae_ensemble'].median(), color='green', linestyle='--', linewidth=2,
               label=f'Median: {df["mae_ensemble"].median():.2f}')
    ax.set_xlabel('Mean Absolute Error (pp)', fontweight='bold')
    ax.set_ylabel('Frequency', fontweight='bold')
    ax.set_title('(d) Distribution of Ensemble MAE', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
else:
    for i, ax in enumerate(axes.flat):
        ax.text(0.5, 0.5, 'Ground truth not available\nfor MAE calculation', 
                ha='center', va='center', fontsize=12, transform=ax.transAxes)
        ax.axis('off')

plt.tight_layout()
plt.savefig('figure2_model_comparison.png', dpi=300, bbox_inches='tight')
plt.savefig('figure2_model_comparison.pdf', dpi=300, bbox_inches='tight')
print("✓ Saved: figure2_model_comparison.png")
print("✓ Saved: figure2_model_comparison.pdf")
plt.close()

# ================================================================
# Figure 3: Ground Truth Comparison
# ================================================================

if has_ground_truth:
    print("\nCreating Figure 3: Ground Truth Comparison...")
    
    fig = plt.figure()
    fig.set_size_inches(14, 10)
    fig.set_dpi(300)
    axes = fig.subplots(2, 2)
    
    # 3a: Scatter plot - Predictions vs Ground Truth
    ax = axes[0, 0]
    stage8 = df[df['stage'] == 8].copy()
    
    ax.scatter(stage8['ground_truth_pi'], stage8['pred_ensemble'], 
              alpha=0.6, s=50, color='steelblue')
    
    # Perfect prediction line
    min_val = min(stage8['ground_truth_pi'].min(), stage8['pred_ensemble'].min())
    max_val = max(stage8['ground_truth_pi'].max(), stage8['pred_ensemble'].max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
    
    # Calculate and show R²
    r, p = pearsonr(stage8['ground_truth_pi'], stage8['pred_ensemble'])
    ax.text(0.05, 0.95, f'R = {r:.3f}\np < 0.001', transform=ax.transAxes, 
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax.set_xlabel('Actual Others\' Willingness (%)', fontweight='bold')
    ax.set_ylabel('Predicted Others\' Willingness (%)', fontweight='bold')
    ax.set_title('(a) Prediction Accuracy (Stage 8)', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3b: MAE distribution by continent
    ax = axes[0, 1]
    continent_mae = df[df['stage'] == 8].groupby('continent')['mae_ensemble'].mean().sort_values()
    
    bars = ax.barh(range(len(continent_mae)), continent_mae.values, 
                   color='lightcoral', alpha=0.7, edgecolor='black')
    ax.set_yticks(range(len(continent_mae)))
    ax.set_yticklabels(continent_mae.index)
    ax.set_xlabel('Mean Absolute Error (pp)', fontweight='bold')
    ax.set_title('(b) MAE by Continent (Stage 8)', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for i, val in enumerate(continent_mae.values):
        ax.text(val, i, f' {val:.2f}', va='center', fontweight='bold')
    
    # 3c: Top 10 best predictions
    ax = axes[1, 0]
    best_countries = stage8.nsmallest(10, 'mae_ensemble')[['countrynew', 'mae_ensemble']]
    
    bars = ax.barh(range(len(best_countries)), best_countries['mae_ensemble'].values, 
                   color='lightgreen', alpha=0.7, edgecolor='black')
    ax.set_yticks(range(len(best_countries)))
    ax.set_yticklabels(best_countries['countrynew'].values)
    ax.set_xlabel('Mean Absolute Error (pp)', fontweight='bold')
    ax.set_title('(c) Top 10 Most Accurate Predictions', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    ax.invert_yaxis()
    
    # Add value labels
    for i, val in enumerate(best_countries['mae_ensemble'].values):
        ax.text(val, i, f' {val:.2f}', va='center', fontweight='bold')
    
    # 3d: Top 10 worst predictions
    ax = axes[1, 1]
    worst_countries = stage8.nlargest(10, 'mae_ensemble')[['countrynew', 'mae_ensemble']]
    
    bars = ax.barh(range(len(worst_countries)), worst_countries['mae_ensemble'].values, 
                   color='lightsalmon', alpha=0.7, edgecolor='black')
    ax.set_yticks(range(len(worst_countries)))
    ax.set_yticklabels(worst_countries['countrynew'].values)
    ax.set_xlabel('Mean Absolute Error (pp)', fontweight='bold')
    ax.set_title('(d) Top 10 Least Accurate Predictions', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    ax.invert_yaxis()
    
    # Add value labels
    for i, val in enumerate(worst_countries['mae_ensemble'].values):
        ax.text(val, i, f' {val:.2f}', va='center', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('figure3_ground_truth_comparison.png', dpi=300, bbox_inches='tight')
    plt.savefig('figure3_ground_truth_comparison.pdf', dpi=300, bbox_inches='tight')
    print("✓ Saved: figure3_ground_truth_comparison.png")
    print("✓ Saved: figure3_ground_truth_comparison.pdf")
    plt.close()

# ================================================================
# Figure 4: Stage 5 Analysis (Own Willingness Impact)
# ================================================================

if has_ground_truth and 'mean_own_willingness' in df.columns:
    print("\nCreating Figure 4: Stage 5 Analysis...")
    
    fig = plt.figure()
    fig.set_size_inches(14, 10)
    fig.set_dpi(300)
    axes = fig.subplots(2, 2)
    
    stage5 = df[df['stage'] == 5].copy()
    stage5['own_willingness_pct'] = stage5['mean_own_willingness'] * 100
    
    # 4a: Own vs Others' willingness
    ax = axes[0, 0]
    ax.scatter(stage5['own_willingness_pct'], stage5['ground_truth_pi'], 
              alpha=0.6, s=50, color='purple')
    
    # Add diagonal line
    min_val = min(stage5['own_willingness_pct'].min(), stage5['ground_truth_pi'].min())
    max_val = max(stage5['own_willingness_pct'].max(), stage5['ground_truth_pi'].max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, 
            label='Own = Others')
    
    r, p = pearsonr(stage5['own_willingness_pct'], stage5['ground_truth_pi'])
    ax.text(0.05, 0.95, f'R = {r:.3f}\np < 0.001', transform=ax.transAxes, 
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax.set_xlabel('Own Willingness (%)', fontweight='bold')
    ax.set_ylabel('Others\' Willingness (%)', fontweight='bold')
    ax.set_title('(a) Own vs Others\' Willingness', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 4b: Prediction vs Own Willingness
    ax = axes[0, 1]
    ax.scatter(stage5['own_willingness_pct'], stage5['pred_ensemble'], 
              alpha=0.6, s=50, color='orange')
    
    r, p = pearsonr(stage5['own_willingness_pct'], stage5['pred_ensemble'])
    ax.text(0.05, 0.95, f'R = {r:.3f}\np < 0.001', transform=ax.transAxes, 
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax.set_xlabel('Own Willingness (%)', fontweight='bold')
    ax.set_ylabel('Predicted Others\' Willingness (%)', fontweight='bold')
    ax.set_title('(b) How Own Willingness Affects Predictions', fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # 4c: Pluralistic ignorance (Own - Others)
    ax = axes[1, 0]
    stage5['pi_gap'] = stage5['own_willingness_pct'] - stage5['ground_truth_pi']
    
    # Sort by PI gap
    pi_sorted = stage5.nlargest(15, 'pi_gap')[['countrynew', 'pi_gap']]
    
    colors = ['lightcoral' if x > 0 else 'lightblue' for x in pi_sorted['pi_gap']]
    bars = ax.barh(range(len(pi_sorted)), pi_sorted['pi_gap'].values, 
                   color=colors, alpha=0.7, edgecolor='black')
    ax.set_yticks(range(len(pi_sorted)))
    ax.set_yticklabels(pi_sorted['countrynew'].values)
    ax.set_xlabel('Pluralistic Ignorance Gap (Own - Others)', fontweight='bold')
    ax.set_title('(c) Top 15 Countries by PI Gap', fontweight='bold')
    ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax.grid(True, alpha=0.3, axis='x')
    ax.invert_yaxis()
    
    # 4d: MAE correlation with PI gap
    ax = axes[1, 1]
    ax.scatter(abs(stage5['pi_gap']), stage5['mae_ensemble'], 
              alpha=0.6, s=50, color='teal')
    
    r, p = pearsonr(abs(stage5['pi_gap']), stage5['mae_ensemble'])
    ax.text(0.05, 0.95, f'R = {r:.3f}\np = {p:.3f}', transform=ax.transAxes, 
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax.set_xlabel('|Pluralistic Ignorance Gap|', fontweight='bold')
    ax.set_ylabel('Mean Absolute Error (pp)', fontweight='bold')
    ax.set_title('(d) Prediction Error vs PI Gap', fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('figure4_stage5_analysis.png', dpi=300, bbox_inches='tight')
    plt.savefig('figure4_stage5_analysis.pdf', dpi=300, bbox_inches='tight')
    print("✓ Saved: figure4_stage5_analysis.png")
    print("✓ Saved: figure4_stage5_analysis.pdf")
    plt.close()

# ================================================================
# Figure 5: Continent Analysis
# ================================================================

if has_ground_truth:
    print("\nCreating Figure 5: Continent Analysis...")
    
    fig = plt.figure()
    fig.set_size_inches(14, 10)
    fig.set_dpi(300)
    axes = fig.subplots(2, 2)
    
    stage8 = df[df['stage'] == 8].copy()
    
    # 5a: MAE by continent (box plot)
    ax = axes[0, 0]
    continents = sorted(stage8['continent'].dropna().unique())
    continent_data = [stage8[stage8['continent'] == c]['mae_ensemble'].values 
                     for c in continents]
    
    bp = ax.boxplot(continent_data, labels=continents, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightcyan')
        patch.set_alpha(0.7)
    
    ax.set_xlabel('Continent', fontweight='bold')
    ax.set_ylabel('Mean Absolute Error (pp)', fontweight='bold')
    ax.set_title('(a) MAE Distribution by Continent', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # 5b: Sample size and mean MAE by continent
    ax = axes[0, 1]
    continent_stats = stage8.groupby('continent').agg({
        'mae_ensemble': ['mean', 'count']
    }).round(2)
    continent_stats.columns = ['Mean MAE', 'N Countries']
    continent_stats = continent_stats.sort_values('Mean MAE')
    
    ax2 = ax.twinx()
    x = range(len(continent_stats))
    width = 0.35
    
    bars1 = ax.bar([i - width/2 for i in x], continent_stats['Mean MAE'], 
                   width, label='Mean MAE', color='lightcoral', alpha=0.7)
    bars2 = ax2.bar([i + width/2 for i in x], continent_stats['N Countries'], 
                    width, label='N Countries', color='lightblue', alpha=0.7)
    
    ax.set_xlabel('Continent', fontweight='bold')
    ax.set_ylabel('Mean MAE (pp)', fontweight='bold', color='darkred')
    ax2.set_ylabel('Number of Countries', fontweight='bold', color='darkblue')
    ax.set_title('(b) Mean MAE and Sample Size by Continent', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(continent_stats.index, rotation=45, ha='right')
    ax.tick_params(axis='y', labelcolor='darkred')
    ax2.tick_params(axis='y', labelcolor='darkblue')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add legends
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
    
    # 5c: Correlation heatmap by continent
    ax = axes[1, 0]
    continent_corr_data = []
    
    for continent in continents:
        cont_data = stage8[stage8['continent'] == continent]
        if len(cont_data) > 2:
            r, _ = pearsonr(cont_data['ground_truth_pi'], cont_data['pred_ensemble'])
            mae = cont_data['mae_ensemble'].mean()
            n = len(cont_data)
            continent_corr_data.append({
                'Continent': continent,
                'Correlation': r,
                'Mean MAE': mae,
                'N': n
            })
    
    corr_df = pd.DataFrame(continent_corr_data).set_index('Continent')
    
    # Create color map based on performance
    colors_list = ['lightgreen' if x < 10 else 'lightyellow' if x < 15 else 'lightcoral' 
                   for x in corr_df['Mean MAE']]
    
    y_pos = range(len(corr_df))
    ax.barh(y_pos, corr_df['Correlation'], color=colors_list, alpha=0.7, edgecolor='black')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(corr_df.index)
    ax.set_xlabel('Prediction Correlation (R)', fontweight='bold')
    ax.set_title('(c) Prediction Accuracy by Continent', fontweight='bold')
    ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='R = 0.5')
    ax.grid(True, alpha=0.3, axis='x')
    ax.legend()
    
    # Add MAE values as text
    for i, (idx, row) in enumerate(corr_df.iterrows()):
        ax.text(row['Correlation'], i, f"  MAE: {row['Mean MAE']:.1f}", 
                va='center', fontsize=9)
    
    # 5d: Best and worst predictions by continent
    ax = axes[1, 1]
    ax.axis('off')
    
    y_pos = 0.95
    ax.text(0.5, y_pos, 'Performance Summary by Continent', 
            ha='center', fontweight='bold', fontsize=12, transform=ax.transAxes)
    y_pos -= 0.08
    
    # Sort by MAE
    sorted_continents = corr_df.sort_values('Mean MAE')
    
    ax.text(0.05, y_pos, 'Best Performance:', fontweight='bold', 
            transform=ax.transAxes, color='green')
    y_pos -= 0.06
    
    for i, (continent, row) in enumerate(sorted_continents.head(3).iterrows()):
        text = f"{i+1}. {continent}: MAE = {row['Mean MAE']:.2f}, R = {row['Correlation']:.3f}, N = {int(row['N'])}"
        ax.text(0.08, y_pos, text, transform=ax.transAxes, fontsize=10)
        y_pos -= 0.05
    
    y_pos -= 0.03
    ax.text(0.05, y_pos, 'Worst Performance:', fontweight='bold', 
            transform=ax.transAxes, color='red')
    y_pos -= 0.06
    
    for i, (continent, row) in enumerate(sorted_continents.tail(3).iterrows()):
        text = f"{i+1}. {continent}: MAE = {row['Mean MAE']:.2f}, R = {row['Correlation']:.3f}, N = {int(row['N'])}"
        ax.text(0.08, y_pos, text, transform=ax.transAxes, fontsize=10)
        y_pos -= 0.05
    
    y_pos -= 0.03
    ax.text(0.05, y_pos, 'Key Insights:', fontweight='bold', 
            transform=ax.transAxes, fontsize=11)
    y_pos -= 0.06
    
    overall_r, _ = pearsonr(stage8['ground_truth_pi'], stage8['pred_ensemble'])
    overall_mae = stage8['mae_ensemble'].mean()
    
    insights = [
        f"• Overall Correlation: R = {overall_r:.3f}",
        f"• Overall Mean MAE: {overall_mae:.2f}",
        f"• Best Continent: {sorted_continents.index[0]} (MAE: {sorted_continents.iloc[0]['Mean MAE']:.2f})",
        f"• Worst Continent: {sorted_continents.index[-1]} (MAE: {sorted_continents.iloc[-1]['Mean MAE']:.2f})"
    ]
    
    for insight in insights:
        ax.text(0.08, y_pos, insight, transform=ax.transAxes, fontsize=9)
        y_pos -= 0.05
    
    plt.tight_layout()
    plt.savefig('figure5_continent_analysis.png', dpi=300, bbox_inches='tight')
    plt.savefig('figure5_continent_analysis.pdf', dpi=300, bbox_inches='tight')
    print("✓ Saved: figure5_continent_analysis.png")
    print("✓ Saved: figure5_continent_analysis.pdf")
    plt.close()

# ================================================================
# Summary
# ================================================================

print("\n" + "="*80)
print("VISUALIZATION COMPLETE!")
print("="*80)
print("\nFigures created:")
print("  - figure1_stage_effects.png / .pdf")
print("  - figure2_model_comparison.png / .pdf")
if has_ground_truth:
    print("  - figure3_ground_truth_comparison.png / .pdf")
    print("  - figure4_stage5_analysis.png / .pdf")
    print("  - figure5_continent_analysis.png / .pdf")
else:
    print("\nNote: Some figures not created due to missing ground truth data.")
print("\n" + "="*80)

VISUALIZATION SCRIPT - MATPLOTLIB 3.9+ COMPATIBLE
Ground truth available: True
Created MAE columns: ['mae_gpt', 'mae_claude', 'mae_gemini', 'mae_llama', 'mae_ensemble']

Creating Figure 1: Stage Effects...
✓ Saved: figure1_stage_effects.png
✓ Saved: figure1_stage_effects.pdf

Creating Figure 2: Model Comparison...
✓ Saved: figure2_model_comparison.png
✓ Saved: figure2_model_comparison.pdf

Creating Figure 3: Ground Truth Comparison...
✓ Saved: figure3_ground_truth_comparison.png
✓ Saved: figure3_ground_truth_comparison.pdf

Creating Figure 4: Stage 5 Analysis...
✓ Saved: figure4_stage5_analysis.png
✓ Saved: figure4_stage5_analysis.pdf

Creating Figure 5: Continent Analysis...
✓ Saved: figure5_continent_analysis.png
✓ Saved: figure5_continent_analysis.pdf

VISUALIZATION COMPLETE!

Figures created:
  - figure1_stage_effects.png / .pdf
  - figure2_model_comparison.png / .pdf
  - figure3_ground_truth_comparison.png / .pdf
  - figure4_stage5_analysis.png / .pdf
  - figure5_continent_analysi

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>